In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
 
np.random.seed(42)
torch.manual_seed(42)
 
STYLES = ["Агрессивный", "Тактический", "Осторожный"]

In [2]:
def generate_player(style):
    if style == "Агрессивный":
        kills = np.random.normal(18, 7)
        deaths = np.random.normal(14, 6)
        accuracy = np.random.normal(38, 14)
        headshot = np.random.normal(15, 10)
        damage = np.random.normal(2600, 850)
        distance = np.random.normal(3200, 1000)
        time_alive = np.random.normal(220, 110)
        shots_fired = np.random.normal(420, 150)
        healing = np.random.normal(80, 60)
 
    elif style == "Тактический":
        kills = np.random.normal(13, 6)
        deaths = np.random.normal(8, 4.5)
        accuracy = np.random.normal(55, 13)
        headshot = np.random.normal(35, 13)
        damage = np.random.normal(2100, 650)
        distance = np.random.normal(2000, 700)
        time_alive = np.random.normal(420, 130)
        shots_fired = np.random.normal(230, 100)
        healing = np.random.normal(150, 85)
 
    else:  # Осторожный
        kills = np.random.normal(6, 4.5)
        deaths = np.random.normal(4, 3)
        accuracy = np.random.normal(45, 15)
        headshot = np.random.normal(20, 11)
        damage = np.random.normal(1100, 500)
        distance = np.random.normal(900, 500)
        time_alive = np.random.normal(600, 160)
        shots_fired = np.random.normal(120, 70)
        healing = np.random.normal(260, 110)
 
    kills = max(kills, 0)
    deaths = max(deaths, 0.5)
    accuracy = np.clip(accuracy, 5, 95)
    headshot = np.clip(headshot, 0, 80)
    damage = max(damage, 50)
    distance = max(distance, 50)
    time_alive = np.clip(time_alive, 20, 900)
    shots_fired = max(shots_fired, 10)
    healing = max(healing, 0)
 
    shots_hit = np.clip(shots_fired * (accuracy / 100) + np.random.normal(0, 8), 0, shots_fired)
 
    matches = np.random.randint(20, 400)
 
    return {
        "Kills": round(kills, 1),
        "Deaths": round(deaths, 1),
        "Accuracy": round(accuracy, 1),
        "HeadshotPercent": round(headshot, 1),
        "DamagePerMatch": round(damage, 1),
        "DistanceTravelled": round(distance, 1),
        "TimeAlive": round(time_alive, 1),
        "ShotsFired": round(shots_fired, 1),
        "ShotsHit": round(shots_hit, 1),
        "HealingUsed": round(healing, 1),
        "MatchesPlayed": matches,
        "PlayStyle": style,
    }
 
 
N = 500
labels = np.random.choice(STYLES, size=N)
df = pd.DataFrame([generate_player(style) for style in labels])

In [3]:
n_swap = int(0.04 * len(df))
swap_idx = np.random.choice(df.index, size=n_swap, replace=False)
for i in swap_idx:
    other_styles = [s for s in STYLES if s != df.loc[i, "PlayStyle"]]
    df.loc[i, "PlayStyle"] = np.random.choice(other_styles)
 
print(df.head())
print("\nСредние значения по стилям:")
print(df.groupby("PlayStyle").mean(numeric_only=True).round(1))

   Kills  Deaths  Accuracy  HeadshotPercent  DamagePerMatch  \
0   20.3     4.3      59.6             37.0          1272.0   
1   21.1    20.8      45.0             25.6          2031.4   
2    5.1     2.5      36.2             29.3          1278.5   
3    0.0     8.2      73.1             38.6           792.4   
4   28.9    11.4      32.3             17.6          2610.3   

   DistanceTravelled  TimeAlive  ShotsFired  ShotsHit  HealingUsed  \
0             1363.1      697.3       192.3     116.2        146.2   
1             2331.9      160.0       229.9     100.1         86.6   
2              553.5      743.9       141.5      56.2        349.4   
3             1091.4      678.7        13.1      13.1        279.9   
4             4130.9      162.4       562.4     188.4         65.6   

   MatchesPlayed    PlayStyle  
0            198   Осторожный  
1            234  Агрессивный  
2            268   Осторожный  
3            287   Осторожный  
4            252  Агрессивный  

Средние

In [4]:
feature_cols = [c for c in df.columns if c != "PlayStyle"]
style_to_idx = {s: i for i, s in enumerate(STYLES)}
 
X = df[feature_cols].values.astype(np.float32)
y = df["PlayStyle"].map(style_to_idx).values.astype(np.int64)
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit только на train!
X_test_scaled = scaler.transform(X_test)
 
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [5]:
class PlayerStyleNet(nn.Module):
    def __init__(self, in_features, activation_layer):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features, 32),
            activation_layer(),
            nn.Linear(32, 16),
            activation_layer(),
            nn.Linear(16, 3),
        )
 
    def forward(self, x):
        return self.network(x)

In [6]:
activations = {
    "Sigmoid": nn.Sigmoid,
    "Tanh": nn.Tanh,
    "ReLU": nn.ReLU,
    "LeakyReLU": nn.LeakyReLU,
    "SiLU": nn.SiLU,
}
 
EPOCHS = 200
results = {}
 
for name, activation in activations.items():
    torch.manual_seed(0)
    model = PlayerStyleNet(len(feature_cols), activation)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()
 
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_loss = criterion(test_logits, y_test_t).item()
        predictions = test_logits.argmax(dim=1)
        accuracy = (predictions == y_test_t).float().mean().item()
 
    results[name] = {"model": model, "loss": test_loss, "accuracy": accuracy}
    print(f"{name:10} | test loss: {test_loss:.4f} | test accuracy: {accuracy * 100:.2f}%")
 
best_name = max(results, key=lambda n: results[n]["accuracy"])
best_model = results[best_name]["model"]
print(f"\nЛучшая функция активации: {best_name} (accuracy={results[best_name]['accuracy'] * 100:.2f}%)")



Sigmoid    | test loss: 0.4045 | test accuracy: 89.00%
Tanh       | test loss: 0.6911 | test accuracy: 85.00%
ReLU       | test loss: 1.3565 | test accuracy: 84.00%
LeakyReLU  | test loss: 1.4087 | test accuracy: 87.00%
SiLU       | test loss: 1.8857 | test accuracy: 84.00%

Лучшая функция активации: Sigmoid (accuracy=89.00%)


In [7]:
def check_player(**kwargs):
    x = np.array([[kwargs[c] for c in feature_cols]], dtype=np.float32)
    x_scaled = scaler.transform(x)
    x_t = torch.tensor(x_scaled, dtype=torch.float32)
 
    best_model.eval()
    with torch.no_grad():
        logits = best_model(x_t)
        probs = torch.softmax(logits, dim=1).numpy()[0]   # логиты -> вероятности
 
    result = {STYLES[i]: round(float(probs[i]) * 100, 1) for i in range(3)}
    return dict(sorted(result.items(), key=lambda item: -item[1]))
 
 
print("\nПример: типичный тактический игрок")
print(check_player(
    Kills=13, Deaths=8, Accuracy=55, HeadshotPercent=35, DamagePerMatch=2100,
    DistanceTravelled=2000, TimeAlive=420, ShotsFired=230, ShotsHit=126,
    HealingUsed=150, MatchesPlayed=200,
))


Пример: типичный тактический игрок
{'Тактический': 99.8, 'Агрессивный': 0.1, 'Осторожный': 0.1}


In [8]:

print("\nНеобычный игрок 1 мало выстрелов, огромный урон")
print(check_player(
    Kills=10, Deaths=3, Accuracy=85, HeadshotPercent=70, DamagePerMatch=2400,
    DistanceTravelled=600, TimeAlive=700, ShotsFired=40, ShotsHit=34,
    HealingUsed=200, MatchesPlayed=150,
))
 
print("\nНеобычный игрок 2 много убийств, но ужасная точность")
print(check_player(
    Kills=25, Deaths=20, Accuracy=15, HeadshotPercent=5, DamagePerMatch=2800,
    DistanceTravelled=3500, TimeAlive=180, ShotsFired=900, ShotsHit=135,
    HealingUsed=30, MatchesPlayed=250,
))
 
print("\nНеобычный игрок 3 почти не играет, только лечится и прячется")
print(check_player(
    Kills=1, Deaths=1, Accuracy=50, HeadshotPercent=10, DamagePerMatch=150,
    DistanceTravelled=400, TimeAlive=850, ShotsFired=15, ShotsHit=7,
    HealingUsed=300, MatchesPlayed=100,
))


Необычный игрок 1 мало выстрелов, огромный урон
{'Осторожный': 62.4, 'Тактический': 34.3, 'Агрессивный': 3.3}

Необычный игрок 2 много убийств, но ужасная точность
{'Агрессивный': 98.1, 'Тактический': 1.3, 'Осторожный': 0.6}

Необычный игрок 3 почти не играет, только лечится и прячется
{'Осторожный': 97.3, 'Агрессивный': 2.3, 'Тактический': 0.4}


In [9]:
rng = np.random.default_rng(0)
mins, maxs = X_train.min(axis=0), X_train.max(axis=0)
 
best_case, best_entropy, best_probs = None, -1, None
 
for _ in range(20000):
    sample = mins + rng.random(len(feature_cols)) * (maxs - mins)
    x_scaled = scaler.transform(sample.reshape(1, -1))
    x_t = torch.tensor(x_scaled, dtype=torch.float32)
 
    with torch.no_grad():
        probs = torch.softmax(best_model(x_t), dim=1).numpy()[0]
 
    entropy = -(probs * np.log(probs + 1e-9)).sum()  
    if entropy > best_entropy:
        best_entropy, best_case, best_probs = entropy, sample, probs
 
print("\nИгрок, на котором модель сомневается больше всего ")
for name, value in zip(feature_cols, best_case):
    print(f"  {name}: {value:.1f}")
print("Вероятности:", {STYLES[i]: round(float(best_probs[i]) * 100, 1) for i in range(3)})


Игрок, на котором модель сомневается больше всего 
  Kills: 14.2
  Deaths: 19.3
  Accuracy: 6.0
  HeadshotPercent: 5.1
  DamagePerMatch: 138.7
  DistanceTravelled: 2780.6
  TimeAlive: 518.8
  ShotsFired: 13.8
  ShotsHit: 70.3
  HealingUsed: 321.7
  MatchesPlayed: 341.4
Вероятности: {'Агрессивный': 36.9, 'Тактический': 30.4, 'Осторожный': 32.7}
